In [ ]:
import requests
import folium
import json
import socket
import ssl
import urllib.parse
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time
import os
from datetime import datetime
import threading
from bs4 import BeautifulSoup
import random

class AdvancedGeoIPRecon:
    def __init__(self, base_path="C:\\Users\\Abdo\\Desktop\\ScanResults"):
        self.base_path = base_path
        self.results = {
            'target_url': '',
            'subdomains': [],
            'exposed_directories': [],
            'ssl_info': {},
            'geo_data': {},
            'screenshot_path': '',
            'threat_map_path': '',
            'scan_time': '',
            'all_files_found': []
        }
        
        # إنشاء مجلد النتائج إذا لم يكن موجوداً
        self.create_results_folder()
    
    def create_results_folder(self):
        """إنشاء مجلد النتائج"""
        if not os.path.exists(self.base_path):
            os.makedirs(self.base_path)
            print(f"✅ تم إنشاء مجلد النتائج: {self.base_path}")
    
    def get_geoip_info(self, domain):
        """الحصول على معلومات الموقع الجغرافي"""
        try:
            # الحصول على IP من الدومين
            ip = socket.gethostbyname(domain)
            
            # استخدام multiple APIs للحصول على معلومات الموقع
            apis = [
                f'http://ipapi.co/{ip}/json/',
                f'http://ip-api.com/json/{ip}',
                f'https://geolocation-db.com/json/{ip}'
            ]
            
            geo_data = {}
            for api_url in apis:
                try:
                    response = requests.get(api_url, timeout=10)
                    if response.status_code == 200:
                        data = response.json()
                        if not geo_data:
                            geo_data = data
                        break
                except:
                    continue
            
            # معالجة البيانات من مختلف الـ APIs
            if 'ipapi.co' in api_url:
                self.results['geo_data'] = {
                    'ip': ip,
                    'city': geo_data.get('city', 'Unknown'),
                    'region': geo_data.get('region', 'Unknown'),
                    'country': geo_data.get('country_name', 'Unknown'),
                    'latitude': geo_data.get('latitude', 0),
                    'longitude': geo_data.get('longitude', 0),
                    'isp': geo_data.get('org', 'Unknown'),
                    'timezone': geo_data.get('timezone', 'Unknown')
                }
            elif 'ip-api.com' in api_url:
                self.results['geo_data'] = {
                    'ip': ip,
                    'city': geo_data.get('city', 'Unknown'),
                    'region': geo_data.get('regionName', 'Unknown'),
                    'country': geo_data.get('country', 'Unknown'),
                    'latitude': geo_data.get('lat', 0),
                    'longitude': geo_data.get('lon', 0),
                    'isp': geo_data.get('isp', 'Unknown'),
                    'timezone': geo_data.get('timezone', 'Unknown')
                }
            else:
                self.results['geo_data'] = {
                    'ip': ip,
                    'city': geo_data.get('city', 'Unknown'),
                    'country': geo_data.get('country_name', 'Unknown'),
                    'latitude': geo_data.get('latitude', 0),
                    'longitude': geo_data.get('longitude', 0),
                    'isp': 'Unknown',
                    'timezone': 'Unknown'
                }
            
            print(f"📍 الموقع الجغرافي: {self.results['geo_data']['city']}, {self.results['geo_data']['country']}")
            print(f"📡 IP: {ip}")
            print(f"🧭 الإحداثيات: {self.results['geo_data']['latitude']}, {self.results['geo_data']['longitude']}")
            print(f"🌐 مزود الخدمة: {self.results['geo_data']['isp']}")
            
        except Exception as e:
            print(f"❌ خطأ في الحصول على الموقع الجغرافي: {e}")
    
    def subdomain_enumeration(self, domain):
        """اكتشاف النطاقات الفرعية باستخدام multiple sources"""
        try:
            print(f"\n🔍 جاري اكتشاف النطاقات الفرعية لـ {domain}...")
            
            subdomains = set()
            
            # 1. crt.sh
            try:
                url = f"https://crt.sh/?q=%25.{domain}&output=json"
                response = requests.get(url, timeout=10)
                if response.status_code == 200:
                    data = response.json()
                    for cert in data:
                        name = cert.get('name_value', '')
                        if name and domain in name:
                            subdomains.update([sub.strip().lower() for sub in name.split('\n') if domain in sub])
            except:
                pass
            
            # 2. hackertarget API
            try:
                url = f"https://api.hackertarget.com/hostsearch/?q={domain}"
                response = requests.get(url, timeout=10)
                if response.status_code == 200:
                    for line in response.text.split('\n'):
                        if ',' in line:
                            subdomain = line.split(',')[0].strip()
                            if domain in subdomain:
                                subdomains.add(subdomain.lower())
            except:
                pass
            
            # 3. sublist3r API (simulated)
            common_subs = [
                f"www.{domain}", f"mail.{domain}", f"ftp.{domain}", 
                f"admin.{domain}", f"blog.{domain}", f"api.{domain}",
                f"test.{domain}", f"dev.{domain}", f"staging.{domain}",
                f"cdn.{domain}", f"static.{domain}", f"media.{domain}",
                f"shop.{domain}", f"store.{domain}", f"app.{domain}",
                f"mobile.{domain}", f"secure.{domain}", f"portal.{domain}"
            ]
            
            for sub in common_subs:
                try:
                    socket.gethostbyname(sub)
                    subdomains.add(sub)
                except:
                    continue
            
            self.results['subdomains'] = sorted(list(subdomains))[:30]  # الحد لـ 30 نطاق
            
            print(f"✅ تم العثور على {len(self.results['subdomains'])} نطاق فرعي:")
            for i, sub in enumerate(self.results['subdomains'][:15], 1):
                print(f"   {i:2d}. {sub}")
            if len(self.results['subdomains']) > 15:
                print(f"   ... و {len(self.results['subdomains']) - 15} أكثر")
                
        except Exception as e:
            print(f"❌ خطأ في اكتشاف النطاقات الفرعية: {e}")
    
    def check_exposed_directories(self, base_url):
        """فحص الدلائل والملفات المكشوفة بشكل شامل"""
        try:
            print(f"\n📁 فحص الدلائل والملفات المكشوفة...")
            
            # قائمة شاملة للدلائل والملفات المهمة
            directories = [
                # Administrative
                '/admin', '/administrator', '/wp-admin', '/dashboard', '/control', 
                '/manager', '/management', '/console', '/backend', '/cpanel',
                '/whm', '/webmail', '/plesk', '/directadmin',
                
                # Backup Files
                '/backup', '/backups', '/backup.zip', '/backup.tar', '/backup.tar.gz',
                '/backup.sql', '/dump.sql', '/database.sql', '/backup.rar',
                '/backup.7z', '/backup.bak', '/backup.old', '/backup.txt',
                '/backup.db', '/backup.dat', '/backup.xml',
                
                # Configuration Files
                '/config', '/configuration', '/config.php', '/config.py', '/config.json',
                '/config.xml', '/settings.py', '/settings.json', '/.env', '/.config',
                '/config.txt', '/configuration.txt', '/config.inc.php', '/config.php.bak',
                '/wp-config.php', '/wp-config.php.bak', '/config.php.save',
                
                # Database Files
                '/db', '/database', '/sql', '/mysql', '/postgresql', '/mongodb',
                '/data', '/data.sql', '/data.db', '/database.db', '/site.db',
                
                # Development Files
                '/.git', '/.svn', '/.hg', '/.DS_Store', '/thumbs.db',
                '/composer.json', '/package.json', '/requirements.txt',
                '/.htaccess', '/web.config', '/robots.txt', '/sitemap.xml',
                
                # Log Files
                '/logs', '/log', '/error.log', '/access.log', '/debug.log',
                '/app.log', '/system.log', '/security.log',
                
                # API and Documentation
                '/api', '/api/v1', '/api/v2', '/graphql', '/swagger', '/swagger-ui',
                '/redoc', '/docs', '/documentation', '/help', '/guide',
                
                # Upload and Media
                '/uploads', '/upload', '/media', '/images', '/files', '/downloads',
                '/assets', '/static', '/public', '/shared',
                
                # Test and Development
                '/test', '/testing', '/dev', '/development', '/staging', '/stage',
                '/demo', '/sandbox', '/playground',
                
                # Security Files
                '/security', '/secure', '/auth', '/authentication', '/login',
                '/signin', '/register', '/signup', '/oauth', '/.well-known',
                
                # System Files
                '/system', '/sys', '/bin', '/lib', '/include', '/src',
                '/vendor', '/node_modules', '/bower_components',
                
                # Temporary Files
                '/tmp', '/temp', '/cache', '/session', '/sessions',
                
                # Custom Business Directories
                '/customer', '/clients', '/users', '/members', '/partners',
                '/orders', '/invoices', '/payments', '/billing', '/account',
                '/profile', '/user', '/member'
            ]
            
            # إضافة امتدادات ملفات شائعة
            file_extensions = ['.zip', '.tar', '.tar.gz', '.sql', '.bak', '.old', '.txt', '.log', '.json', '.xml']
            base_dirs = ['/backup', '/db', '/database', '/config', '/log', '/tmp']
            
            for base_dir in base_dirs:
                for ext in file_extensions:
                    directories.append(f"{base_dir}{ext}")
                    directories.append(f"{base_dir}1{ext}")
                    directories.append(f"{base_dir}2{ext}")
            
            exposed = []
            all_files = []
            
            print("   🔎 جاري فحص 150+ مسار...")
            
            for i, directory in enumerate(directories[:150]):  # فحص أول 150 مسار للسرعة
                test_url = f"{base_url.rstrip('/')}{directory}"
                try:
                    response = requests.get(test_url, timeout=3, allow_redirects=True, verify=False)
                    
                    file_info = {
                        'path': directory,
                        'url': test_url,
                        'status': response.status_code,
                        'size': len(response.content),
                        'content_type': response.headers.get('content-type', 'Unknown')
                    }
                    
                    all_files.append(file_info)
                    
                    # اعتبار الملف مكشوف إذا كانت الحالة 200 أو 301 أو 302 أو 403
                    if response.status_code in [200, 301, 302, 403]:
                        # تصفية النتائج المهمة فقط
                        if (response.status_code == 200 and len(response.content) > 100) or \
                           any(keyword in directory.lower() for keyword in ['admin', 'backup', 'config', '.git', '.env', 'sql', 'log']):
                            exposed.append(file_info)
                            
                            status_icon = "🟢" if response.status_code == 200 else "🟡"
                            print(f"   {status_icon} مكشوف: {directory} (الحالة: {response.status_code}, الحجم: {len(response.content)} bytes)")
                    
                except requests.exceptions.Timeout:
                    continue
                except Exception as e:
                    continue
            
            self.results['exposed_directories'] = exposed
            self.results['all_files_found'] = all_files
            
            print(f"✅ تم فحص {len(all_files)} مسار، وُجد {len(exposed)} مسار مكشوف")
            
        except Exception as e:
            print(f"❌ خطأ في فحص الدلائل: {e}")
    
    def take_screenshot(self, url):
        """التقاط صورة للصفحة"""
        try:
            print(f"\n📸 جاري التقاط صورة للصفحة...")
            
            # إعداد متصفح Chrome
            chrome_options = Options()
            chrome_options.add_argument("--headless")
            chrome_options.add_argument("--no-sandbox")
            chrome_options.add_argument("--disable-dev-shm-usage")
            chrome_options.add_argument("--window-size=1920,1080")
            chrome_options.add_argument("--disable-gpu")
            chrome_options.add_argument("--disable-extensions")
            
            driver = webdriver.Chrome(options=chrome_options)
            driver.get(url)
            time.sleep(5)  # انتظار أطول لتحميل الصفحة
            
            # حفظ الصورة في المسار المحدد
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            screenshot_filename = f"screenshot_{timestamp}.png"
            screenshot_path = os.path.join(self.base_path, screenshot_filename)
            driver.save_screenshot(screenshot_path)
            driver.quit()
            
            self.results['screenshot_path'] = screenshot_path
            print(f"✅ تم حفظ الصورة في: {screenshot_path}")
            
        except Exception as e:
            print(f"❌ خطأ في التقاط الصورة: {e}")
            # محاولة بديلة باستخدام requests
            try:
                response = requests.get(url, timeout=10)
                if response.status_code == 200:
                    print("   ℹ️  الصفحة متاحة ولكن لا يمكن التقاط لقطة شاشة")
            except:
                print("   ❌ لا يمكن الوصول إلى الصفحة")
    
    def check_ssl_tls(self, domain):
        """فحص إعدادات SSL/TLS"""
        try:
            print(f"\n🔒 فحص شهادة SSL/TLS...")
            
            context = ssl.create_default_context()
            with socket.create_connection((domain, 443), timeout=10) as sock:
                with context.wrap_socket(sock, server_hostname=domain) as ssock:
                    cert = ssock.getpeercert()
                    
                    # معلومات الشهادة
                    ssl_info = {
                        'subject': dict(x[0] for x in cert['subject']),
                        'issuer': dict(x[0] for x in cert['issuer']),
                        'version': cert.get('version', 'Unknown'),
                        'not_before': cert['notBefore'],
                        'not_after': cert['notAfter'],
                        'san': cert.get('subjectAltName', []),
                        'serialNumber': cert.get('serialNumber', 'Unknown'),
                        'protocol': ssock.version()
                    }
                    
                    self.results['ssl_info'] = ssl_info
                    
                    print(f"✅ الشهادة صادرة من: {ssl_info['issuer'].get('organizationName', 'Unknown')}")
                    print(f"📅 صالحة من: {ssl_info['not_before']}")
                    print(f"📅 تنتهي في: {ssl_info['not_after']}")
                    print(f"🔐 البروتوكول: {ssl_info['protocol']}")
                    
                    # فحص إذا كانت الشهادة منتهية
                    from datetime import datetime
                    expiry_date = datetime.strptime(ssl_info['not_after'], '%b %d %H:%M:%S %Y %Z')
                    if expiry_date < datetime.now():
                        print("⚠️  تحذير: شهادة SSL منتهية الصلاحية!")
                    
        except Exception as e:
            print(f"❌ خطأ في فحص SSL: {e}")
    
    def create_threat_map(self):
        """إنشاء خريطة التهديدات التفاعلية"""
        try:
            print(f"\n🗺️ جاري إنشاء خريطة التهديدات...")
            
            # إنشاء الخريطة الأساسية
            if self.results['geo_data'].get('latitude') and self.results['geo_data'].get('longitude'):
                threat_map = folium.Map(
                    location=[
                        self.results['geo_data']['latitude'],
                        self.results['geo_data']['longitude']
                    ],
                    zoom_start=12,
                    tiles='OpenStreetMap'
                )
            else:
                # موقع افتراضي إذا لم يتوفر الموقع
                threat_map = folium.Map(location=[24.7136, 46.6753], zoom_start=5)
            
            # إضافة النقطة الرئيسية
            if self.results['geo_data']:
                folium.Marker(
                    [self.results['geo_data']['latitude'], self.results['geo_data']['longitude']],
                    popup=f"""
                    <div style="width: 250px">
                    <h3>🎯 الموقع المستهدف</h3>
                    <b>URL:</b> {self.results['target_url']}<br>
                    <b>IP:</b> {self.results['geo_data']['ip']}<br>
                    <b>المدينة:</b> {self.results['geo_data']['city']}<br>
                    <b>البلد:</b> {self.results['geo_data']['country']}<br>
                    <b>مزود الخدمة:</b> {self.results['geo_data']['isp']}<br>
                    <b>الإحداثيات:</b> {self.results['geo_data']['latitude']:.4f}, {self.results['geo_data']['longitude']:.4f}
                    </div>
                    """,
                    tooltip="🎯 الموقع الرئيسي",
                    icon=folium.Icon(color='red', icon='flag', prefix='fa')
                ).add_to(threat_map)
            
            # إضافة النطاقات الفرعية على الخريطة
            for i, subdomain in enumerate(self.results['subdomains'][:8]):
                try:
                    sub_ip = socket.gethostbyname(subdomain)
                    # إحداثيات عشوائية قريبة من الموقع الرئيسي للتمثيل
                    lat = self.results['geo_data'].get('latitude', 24.7136) + random.uniform(-0.3, 0.3)
                    lon = self.results['geo_data'].get('longitude', 46.6753) + random.uniform(-0.3, 0.3)
                    
                    folium.Marker(
                        [lat, lon],
                        popup=f"<b>🌐 نطاق فرعي:</b> {subdomain}<br><b>IP:</b> {sub_ip}",
                        tooltip=f"🌐 {subdomain}",
                        icon=folium.Icon(color='blue', icon='globe', prefix='fa')
                    ).add_to(threat_map)
                    
                    # إضافة خط connecting
                    folium.PolyLine(
                        [[self.results['geo_data']['latitude'], self.results['geo_data']['longitude']], 
                         [lat, lon]],
                        color="blue",
                        weight=1,
                        opacity=0.5,
                        popup=f"اتصال: {self.results['target_url']} → {subdomain}"
                    ).add_to(threat_map)
                    
                except:
                    continue
            
            # إضافة دائرة نصف قطرها
            folium.Circle(
                location=[self.results['geo_data']['latitude'], self.results['geo_data']['longitude']],
                radius=5000,  # 5km
                popup="نطاق التهديد المحتمل",
                color="red",
                fill=True,
                fillOpacity=0.1
            ).add_to(threat_map)
            
            # حفظ الخريطة
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            map_filename = f"threat_map_{timestamp}.html"
            map_path = os.path.join(self.base_path, map_filename)
            threat_map.save(map_path)
            
            self.results['threat_map_path'] = map_path
            print(f"✅ تم حفظ خريطة التهديدات في: {map_path}")
            
        except Exception as e:
            print(f"❌ خطأ في إنشاء خريطة التهديدات: {e}")
    
    def save_results_to_file(self):
        """حفظ جميع النتائج في ملف"""
        try:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            report_filename = f"scan_report_{timestamp}.txt"
            report_path = os.path.join(self.base_path, report_filename)
            
            with open(report_path, 'w', encoding='utf-8') as f:
                f.write("=" * 70 + "\n")
                f.write("📊 تقرير المسح الأمني الشامل\n")
                f.write("=" * 70 + "\n\n")
                
                f.write(f"🎯 الهدف: {self.results['target_url']}\n")
                f.write(f"🕐 وقت المسح: {self.results['scan_time']}\n\n")
                
                f.write("📍 معلومات الموقع الجغرافي:\n")
                f.write("-" * 40 + "\n")
                for key, value in self.results['geo_data'].items():
                    f.write(f"  {key}: {value}\n")
                f.write("\n")
                
                f.write(f"🌐 النطاقات الفرعية ({len(self.results['subdomains'])}):\n")
                f.write("-" * 40 + "\n")
                for i, sub in enumerate(self.results['subdomains'], 1):
                    f.write(f"  {i:2d}. {sub}\n")
                f.write("\n")
                
                f.write(f"📁 الدلائل والملفات المكشوفة ({len(self.results['exposed_directories'])}):\n")
                f.write("-" * 40 + "\n")
                for i, dir_info in enumerate(self.results['exposed_directories'], 1):
                    f.write(f"  {i:2d}. {dir_info['path']} (الحالة: {dir_info['status']}, الحجم: {dir_info['size']} bytes)\n")
                f.write("\n")
                
                f.write("🔒 معلومات SSL/TLS:\n")
                f.write("-" * 40 + "\n")
                if self.results['ssl_info']:
                    for key, value in self.results['ssl_info'].items():
                        if key != 'san':  # تخطي SAN لعدم الإطالة
                            f.write(f"  {key}: {value}\n")
                f.write("\n")
                
                f.write("📁 الملفات المحفوظة:\n")
                f.write("-" * 40 + "\n")
                f.write(f"  📸 لقطة الشاشة: {self.results['screenshot_path']}\n")
                f.write(f"  🗺️ خريطة التهديدات: {self.results['threat_map_path']}\n")
                f.write(f"  📄 هذا التقرير: {report_path}\n")
            
            print(f"💾 تم حفظ التقرير الكامل في: {report_path}")
            return report_path
            
        except Exception as e:
            print(f"❌ خطأ في حفظ التقرير: {e}")
    
    def generate_report(self):
        """توليد تقرير مفصل في الكونسول"""
        print("\n" + "="*70)
        print("📊 التقرير النهائي - نتائج المسح الأمني")
        print("="*70)
        
        print(f"🎯 الهدف: {self.results['target_url']}")
        print(f"🕐 وقت المسح: {self.results['scan_time']}")
        print(f"📍 الموقع: {self.results['geo_data'].get('city', 'Unknown')}, {self.results['geo_data'].get('country', 'Unknown')}")
        print(f"📡 IP: {self.results['geo_data'].get('ip', 'Unknown')}")
        print(f"🧭 الإحداثيات: {self.results['geo_data'].get('latitude', 'Unknown')}, {self.results['geo_data'].get('longitude', 'Unknown')}")
        print(f"🌐 مزود الخدمة: {self.results['geo_data'].get('isp', 'Unknown')}")
        
        print(f"\n🌐 النطاقات الفرعية ({len(self.results['subdomains'])}):")
        for i, sub in enumerate(self.results['subdomains'][:10], 1):
            print(f"   {i:2d}. {sub}")
        if len(self.results['subdomains']) > 10:
            print(f"   ... و {len(self.results['subdomains']) - 10} أكثر")
        
        print(f"\n📁 الدلائل المكشوفة ({len(self.results['exposed_directories'])}):")
        for i, dir_info in enumerate(self.results['exposed_directories'][:10], 1):
            status_icon = "🟢" if dir_info['status'] == 200 else "🟡"
            print(f"   {status_icon} {dir_info['path']} (الحالة: {dir_info['status']})")
        if len(self.results['exposed_directories']) > 10:
            print(f"   ... و {len(self.results['exposed_directories']) - 10} أكثر")
        
        print(f"\n🔒 معلومات SSL:")
        if self.results['ssl_info']:
            print(f"   - المصدر: {self.results['ssl_info'].get('issuer', {}).get('organizationName', 'Unknown')}")
            print(f"   - الصلاحية: {self.results['ssl_info'].get('not_after', 'Unknown')}")
            print(f"   - البروتوكول: {self.results['ssl_info'].get('protocol', 'Unknown')}")
        
        print(f"\n📁 الملفات المحفوظة في: {self.base_path}")
        print(f"   - 📸 لقطة الشاشة: {os.path.basename(self.results['screenshot_path'])}")
        print(f"   - 🗺️ خريطة التهديدات: {os.path.basename(self.results['threat_map_path'])}")
    
    def run_full_scan(self, url):
        """تشغيل المسح الكامل"""
        print(f"🚀 بدء المسح الشامل لـ: {url}")
        start_time = time.time()
        
        # تنظيف URL
        if not url.startswith(('http://', 'https://')):
            url = 'https://' + url
        
        self.results['target_url'] = url
        self.results['scan_time'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        parsed_url = urllib.parse.urlparse(url)
        domain = parsed_url.netloc
        
        # تنفيذ جميع الوظائف
        self.get_geoip_info(domain)
        self.subdomain_enumeration(domain)
        self.check_exposed_directories(url)
        self.check_ssl_tls(domain)
        self.take_screenshot(url)
        self.create_threat_map()
        
        # حساب وقت التنفيذ
        end_time = time.time()
        execution_time = end_time - start_time
        
        print(f"\n⏱️  وقت التنفيذ: {execution_time:.2f} ثانية")
        
        # عرض التقرير وحفظه
        self.generate_report()
        report_path = self.save_results_to_file()
        
        print(f"\n🎉 تم الانتهاء من المسح!")
        print(f"💾 جميع النتائج محفوظة في: {self.base_path}")
        print(f"📄 التقرير الكامل: {report_path}")

def main():
    """الدالة الرئيسية"""
    print("🌐 GeoIP + URL Recon Module - الإصدار المتقدم")
    print("=" * 60)
    print("أدخل URL لبدء المسح الأمني الشامل")
    print("=" * 60)
    
    # المسار الافتراضي
    default_path = "C:\\Users\\Abdo\\Desktop\\ScanResults"
    
    url = input("🎯 أدخل الـ URL (مثال: example.com): ").strip()
    
    if not url:
        print("❌ يجب إدخال URL صالح")
        return
    
    # إنشاء كائن المسح وتشغيله
    scanner = AdvancedGeoIPRecon(default_path)
    scanner.run_full_scan(url)

if __name__ == "__main__":
    main()

🌐 GeoIP + URL Recon Module - الإصدار المتقدم
أدخل URL لبدء المسح الأمني الشامل


🎯 أدخل الـ URL (مثال: example.com):  https://dulms.deltauniv.edu.eg/login.aspx


🚀 بدء المسح الشامل لـ: https://dulms.deltauniv.edu.eg/login.aspx
📍 الموقع الجغرافي: Giza, Egypt
📡 IP: 45.240.88.14
🧭 الإحداثيات: 30.0046, 31.2044
🌐 مزود الخدمة: LINKdotNET AS number

🔍 جاري اكتشاف النطاقات الفرعية لـ dulms.deltauniv.edu.eg...
✅ تم العثور على 4 نطاق فرعي:
    1. dulms.deltauniv.edu.eg
    2. ftp.dulms.deltauniv.edu.eg
    3. mail.dulms.deltauniv.edu.eg
    4. www.dulms.deltauniv.edu.eg

📁 فحص الدلائل والملفات المكشوفة...
   🔎 جاري فحص 150+ مسار...


C:\Users\Abdo\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'dulms.deltauniv.edu.eg'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\Abdo\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'dulms.deltauniv.edu.eg'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


   🟢 مكشوف: /admin (الحالة: 200, الحجم: 8145 bytes)
   🟢 مكشوف: /administrator (الحالة: 200, الحجم: 8153 bytes)


...

   🟢 مكشوف: /member (الحالة: 200, الحجم: 8146 bytes)
   🟢 مكشوف: /backup.zip (الحالة: 200, الحجم: 8150 bytes)


C:\Users\Abdo\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'dulms.deltauniv.edu.eg'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\Abdo\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'dulms.deltauniv.edu.eg'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


   🟢 مكشوف: /backup1.zip (الحالة: 200, الحجم: 8151 bytes)
   🟢 مكشوف: /backup2.zip (الحالة: 200, الحجم: 8151 bytes)


C:\Users\Abdo\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'dulms.deltauniv.edu.eg'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\Abdo\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'dulms.deltauniv.edu.eg'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


   🟢 مكشوف: /backup.tar (الحالة: 200, الحجم: 8150 bytes)
   🟢 مكشوف: /backup1.tar (الحالة: 200, الحجم: 8151 bytes)
✅ تم فحص 150 مسار، وُجد 150 مسار مكشوف

🔒 فحص شهادة SSL/TLS...
✅ الشهادة صادرة من: Starfield Technologies, Inc.
📅 صالحة من: May 13 11:58:00 2025 GMT
📅 تنتهي في: May 20 11:26:11 2026 GMT
🔐 البروتوكول: TLSv1.2

📸 جاري التقاط صورة للصفحة...
✅ تم حفظ الصورة في: C:\Users\Abdo\Desktop\ScanResults\screenshot_20251025_144627.png

🗺️ جاري إنشاء خريطة التهديدات...
✅ تم حفظ خريطة التهديدات في: C:\Users\Abdo\Desktop\ScanResults\threat_map_20251025_144629.html

⏱️  وقت التنفيذ: 58.11 ثانية

📊 التقرير النهائي - نتائج المسح الأمني
🎯 الهدف: https://dulms.deltauniv.edu.eg/login.aspx
🕐 وقت المسح: 2025-10-25 14:45:31
📍 الموقع: Giza, Egypt
📡 IP: 45.240.88.14
🧭 الإحداثيات: 30.0046, 31.2044
🌐 مزود الخدمة: LINKdotNET AS number

🌐 النطاقات الفرعية (4):
    1. dulms.deltauniv.edu.eg
    2. ftp.dulms.deltauniv.edu.eg
    3. mail.dulms.deltauniv.edu.eg
    4. www.dulms.deltauniv.edu.eg

📁 الدلائل ال

In [1]:
!pip install requests==2.31.0 folium==0.15.1 selenium==4.15.0 beautifulsoup4==4.12.2 urllib3==1.26.18 pillow==10.1.0 cryptography==41.0.7 sslscan==2.0.0 webdriver-manager==4.0.1

     ---------------------------------------- 0.0/50.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/50.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/50.8 MB ? eta -:--:--
     ---------------------------------------- 0.3/50.8 MB ? eta -:--:--
     ---------------------------------------- 0.3/50.8 MB ? eta -:--:--
     --------------------------------------- 0.5/50.8 MB 556.2 kB/s eta 0:01:31
     --------------------------------------- 0.5/50.8 MB 556.2 kB/s eta 0:01:31
     --------------------------------------- 0.5/50.8 MB 556.2 kB/s eta 0:01:31
      -------------------------------------- 0.8/50.8 MB 537.2 kB/s eta 0:01:34
      -------------------------------------- 0.8/50.8 MB 537.2 kB/s eta 0:01:34
      -------------------------------------- 1.0/50.8 MB 522.5 kB/s eta 0:01:36
      -------------------------------------- 1.0/50.8 MB 522.5 kB/s eta 0:01:36
     - ------------------------------------- 1.3/50.8 MB 524.1 kB/s eta 0:01:35


  error: subprocess-exited-with-error
  
  Getting requirements to build wheel did not run successfully.
  exit code: 1
  
  [21 lines of output]
  Traceback (most recent call last):
    File "C:\Users\Abdo\anaconda3\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 389, in <module>
      main()
      ~~~~^^
    File "C:\Users\Abdo\anaconda3\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 373, in main
      json_out["return_val"] = hook(**hook_input["kwargs"])
                               ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
    File "C:\Users\Abdo\anaconda3\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 143, in get_requires_for_build_wheel
      return hook(config_settings)
    File "C:\Users\Abdo\AppData\Local\Temp\pip-build-env-7ogylz8c\overlay\Lib\site-packages\setuptools\build_meta.py", line 331, in get_requires_for_build_wheel
      return self._get_build_requires(config_settings, requirement

In [3]:
!pip install selenium

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.7 MB 448.0 kB/s eta 0:00:21
   -- ------------------------------------- 0.5/9.7 MB 448.0 kB/s eta 0:00:21
   -- ------------------------------------- 0.5/9.7 MB 448.0 kB/s eta 0:00:21
   --- ------------------------------------ 0.8/9.7 MB 463.2 kB/s eta 0:00:20
   --- ------------------------------------ 0.8/9.7 MB 463.2 kB/s eta 0:00:20
   ---- ----------------------------------- 1.0/9.7 MB 479.2 kB/s eta 0:00:19
   ---- ----------------------------------- 1.0/9.7 MB 479.2 kB/s eta 0:00:19
   ---- --------------------------------